In [3]:
import os
import json
import numpy as np
import pandas as pd

# ===== Scikit-learn preprocessing =====
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# ===== Utilities =====
import joblib

In [5]:
import os, json, pandas as pd, numpy as np

SPLIT_DIR = "../outputs/splits"
manifest_path = os.path.join(SPLIT_DIR, "data_manifest.json")

assert os.path.exists(manifest_path), "Missing data_manifest.json. Re-run notebook 02."

with open(manifest_path, "r", encoding="utf-8") as f:
    manifest = json.load(f)

files_to_use = manifest["files_used"]
SAMPLE_FRAC = manifest.get("sample_frac", 0.15)
RANDOM_STATE = manifest.get("random_state", 42)
data_mode = manifest.get("data_mode", "unknown")

print(f"Loading dataset from manifest (sample_frac={SAMPLE_FRAC}):")
print(f"Strategy: Load in chunks with dtype optimization to avoid OOM")

df_list = []
for i, f in enumerate(files_to_use, 1):
    print(f"  [{i}/{len(files_to_use)}] {os.path.basename(f)}")
    
    # Memory-efficient loading: chunked read + sample
    chunk_list = []
    for chunk in pd.read_csv(f, chunksize=50000, low_memory=False):
        # Sample each chunk to reduce memory
        chunk_sampled = chunk.sample(frac=SAMPLE_FRAC, random_state=RANDOM_STATE)
        chunk_list.append(chunk_sampled)
    
    if chunk_list:
        df_part = pd.concat(chunk_list, ignore_index=True)
        df_list.append(df_part)

df = pd.concat(df_list, ignore_index=True)

print("\nDataset loaded")
print("Mode:", data_mode)
print("Rows:", df.shape[0], "| expected:", manifest["n_rows"])

# Allow small difference due to chunking + sampling
row_diff = abs(len(df) - int(manifest["n_rows"]))
if row_diff > 100:
    print(f"⚠️  Warning: Row count mismatch > 100 rows (diff={row_diff})")
else:
    print(f"✅ Row count close enough (diff={row_diff})")

Loading dataset from manifest (sample_frac=0.15):
Strategy: Load in chunks with dtype optimization to avoid OOM
  [1/23] Network_dataset_1.csv
  [2/23] Network_dataset_10.csv
  [3/23] Network_dataset_11.csv
  [4/23] Network_dataset_12.csv
  [5/23] Network_dataset_13.csv
  [6/23] Network_dataset_14.csv
  [7/23] Network_dataset_15.csv
  [8/23] Network_dataset_16.csv
  [9/23] Network_dataset_17.csv
  [10/23] Network_dataset_18.csv
  [11/23] Network_dataset_19.csv
  [12/23] Network_dataset_2.csv
  [13/23] Network_dataset_20.csv
  [14/23] Network_dataset_21.csv
  [15/23] Network_dataset_22.csv
  [16/23] Network_dataset_23.csv
  [17/23] Network_dataset_3.csv
  [18/23] Network_dataset_4.csv
  [19/23] Network_dataset_5.csv
  [20/23] Network_dataset_6.csv
  [21/23] Network_dataset_7.csv
  [22/23] Network_dataset_8.csv
  [23/23] Network_dataset_9.csv

Dataset loaded
Mode: processed_stratified_sample_23files_frac0.15
Rows: 3350853 | expected: 3350853
✅ Row count close enough (diff=0)


In [6]:
SPLIT_DIR = "../outputs/splits"

train_idx = np.load(os.path.join(SPLIT_DIR, "train_idx.npy"))
val_idx   = np.load(os.path.join(SPLIT_DIR, "val_idx.npy"))
test_idx  = np.load(os.path.join(SPLIT_DIR, "test_idx.npy"))

# sanity check bounds
n = len(df)
assert train_idx.max() < n and val_idx.max() < n and test_idx.max() < n, "Split index out of bounds!"

df_train = df.iloc[train_idx].copy()
df_val   = df.iloc[val_idx].copy()
df_test  = df.iloc[test_idx].copy()

print("Loaded splits:")
print("Train:", df_train.shape, "Val:", df_val.shape, "Test:", df_test.shape)

Loaded splits:
Train: (2345597, 47) Val: (502628, 47) Test: (502628, 47)


In [7]:
assert "label" in df.columns, "label not found"

DROP_COLUMNS = ["src_ip", "dst_ip", "type"]  # leakage-prone / metadata
for c in DROP_COLUMNS:
    assert c in df.columns, f"Expected column missing: {c}"

y_train = df_train["label"].astype(int)
y_val   = df_val["label"].astype(int)
y_test  = df_test["label"].astype(int)

X_train_raw = df_train.drop(columns=["label"] + DROP_COLUMNS)
X_val_raw   = df_val.drop(columns=["label"] + DROP_COLUMNS)
X_test_raw  = df_test.drop(columns=["label"] + DROP_COLUMNS)

print("After dropping leakage cols:")
print("X_train:", X_train_raw.shape, "X_val:", X_val_raw.shape, "X_test:", X_test_raw.shape)


After dropping leakage cols:
X_train: (2345597, 43) X_val: (502628, 43) X_test: (502628, 43)


In [8]:
NUMERIC_FEATURES = [
    "duration",
    "src_bytes", "dst_bytes",
    "src_pkts", "dst_pkts",
    "src_ip_bytes", "dst_ip_bytes",
    "missed_bytes",
    "src_port", "dst_port",
    "http_request_body_len",
    "http_response_body_len",
]

CATEGORICAL_FEATURES = [
    "proto",
    "service",
    "conn_state",
    "http_method",
    "http_version",
    "http_status_code",
    "ssl_version",
    "ssl_cipher",
    "weird_name",
]

BOOLEAN_FEATURES = [
    "dns_AA", "dns_RD", "dns_RA",
    "ssl_resumed", "ssl_established",
]


In [9]:
all_feats = set(NUMERIC_FEATURES + CATEGORICAL_FEATURES + BOOLEAN_FEATURES)

missing = [c for c in all_feats if c not in X_train_raw.columns]
assert len(missing) == 0, f"Feature(s) not found in data: {missing}"

print("Feature group sizes:")
print("Numeric:", len(NUMERIC_FEATURES))
print("Categorical:", len(CATEGORICAL_FEATURES))
print("Boolean:", len(BOOLEAN_FEATURES))


Feature group sizes:
Numeric: 12
Categorical: 9
Boolean: 5


In [10]:
def replace_dash_with_none(X: pd.DataFrame) -> pd.DataFrame:
    X = X.copy()
    for col in X.columns:
        X[col] = X[col].astype(str).replace("-", "NONE")
    return X

def cast_boolean_like(X: pd.DataFrame) -> pd.DataFrame:
    """
    Robust boolean casting:
    - "-" -> "NONE" -> 0 by default
    - "T","True","1" -> 1
    - "F","False","0","NONE" -> 0
    If values are already numeric 0/1, keep them.
    """
    X = X.copy()
    for col in X.columns:
        s = X[col]

        # if numeric already (0/1), just coerce
        if pd.api.types.is_numeric_dtype(s):
            X[col] = s.fillna(0).astype(int)
            continue

        s = s.astype(str).replace("-", "NONE").str.lower()
        true_set  = {"t", "true", "1", "yes"}
        false_set = {"f", "false", "0", "none", "nan", ""}

        X[col] = s.apply(lambda v: 1 if v in true_set else 0).astype(int)

    return X


In [11]:
def cast_boolean_like_df(df_, bool_cols):
    df_ = df_.copy()
    for col in bool_cols:
        s = df_[col]
        if pd.api.types.is_numeric_dtype(s):
            df_[col] = s.fillna(0).astype(int)
            continue
        s = s.astype(str).replace("-", "NONE").str.lower()
        true_set = {"t", "true", "1", "yes"}
        df_[col] = s.apply(lambda v: 1 if v in true_set else 0).astype(int)
    return df_

X_train_raw = cast_boolean_like_df(X_train_raw, BOOLEAN_FEATURES)
X_val_raw   = cast_boolean_like_df(X_val_raw, BOOLEAN_FEATURES)
X_test_raw  = cast_boolean_like_df(X_test_raw, BOOLEAN_FEATURES)

print("Casted boolean-like features to {0,1}.")


Casted boolean-like features to {0,1}.


In [12]:
numeric_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

boolean_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
])


preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, NUMERIC_FEATURES),
        ("cat", categorical_pipe, CATEGORICAL_FEATURES),
        ("bool", boolean_pipe, BOOLEAN_FEATURES),
    ],
    remainder="drop",
    verbose_feature_names_out=False
)

print(preprocess)

ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['duration', 'src_bytes', 'dst_bytes',
                                  'src_pkts', 'dst_pkts', 'src_ip_bytes',
                                  'dst_ip_bytes', 'missed_bytes', 'src_port',
                                  'dst_port', 'http_request_body_len',
                                  'http_response_body_len']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImpu...
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False))]),
                                 

In [13]:
# Normalize dashes to 'NONE' in categorical features before pipeline
def normalize_dashes(df_in):
    df_out = df_in.copy()
    for col in CATEGORICAL_FEATURES:
        if col in df_out.columns:
            df_out[col] = df_out[col].astype(str).replace("-", "NONE")
    return df_out

X_train_raw = normalize_dashes(X_train_raw)
X_val_raw = normalize_dashes(X_val_raw)
X_test_raw = normalize_dashes(X_test_raw)

print("Normalized dashes in categorical features")


Normalized dashes in categorical features


In [14]:
DASH_AS_NONE_COLS = CATEGORICAL_FEATURES + ["dns_query", "http_uri", "http_user_agent", 
                                            "http_orig_mime_types", "http_resp_mime_types",
                                            "weird_addl", "weird_notice",
                                            "ssl_subject", "ssl_issuer"]

# Keep only those that exist (robust)
DASH_AS_NONE_COLS = [c for c in DASH_AS_NONE_COLS if c in X_train_raw.columns]

def normalize_dash(df_):
    df_ = df_.copy()
    for c in DASH_AS_NONE_COLS:
        df_[c] = df_[c].astype(str).replace("-", "NONE")
    return df_

X_train_raw = normalize_dash(X_train_raw)
X_val_raw   = normalize_dash(X_val_raw)
X_test_raw  = normalize_dash(X_test_raw)

print("Normalized '-' to 'NONE' for categorical-like columns:", len(DASH_AS_NONE_COLS))


Normalized '-' to 'NONE' for categorical-like columns: 18


In [15]:
# 1) (Quan trọng) Drop đúng các cột không dùng / leakage
DROP_COLS = ["src_ip", "dst_ip", "type", "ts"]  # stage1 bắt buộc drop
# (optional nhưng khuyên drop vì high-cardinality / không dùng stage1)
DROP_COLS += ["http_uri", "http_referrer", "http_user_agent", "ssl_subject", "ssl_issuer", "weird_addl", "weird_notice", "dns_query"]

for c in DROP_COLS:
    if c in X_train_raw.columns:
        pass

X_train_raw = X_train_raw.drop(columns=[c for c in DROP_COLS if c in X_train_raw.columns], errors="ignore")
X_val_raw   = X_val_raw.drop(columns=[c for c in DROP_COLS if c in X_val_raw.columns], errors="ignore")
X_test_raw  = X_test_raw.drop(columns=[c for c in DROP_COLS if c in X_test_raw.columns], errors="ignore")

print("After drop:", X_train_raw.shape, X_val_raw.shape, X_test_raw.shape)

# 2) Check numeric cols nào đang bị object -> ép về numeric
bad_numeric = [c for c in NUMERIC_FEATURES if c in X_train_raw.columns and X_train_raw[c].dtype == "object"]
print("Numeric cols with object dtype:", bad_numeric)

for c in bad_numeric:
    # ép kiểu: string không convert được -> NaN, median imputer sẽ xử lý
    X_train_raw[c] = pd.to_numeric(X_train_raw[c], errors="coerce")
    X_val_raw[c]   = pd.to_numeric(X_val_raw[c], errors="coerce")
    X_test_raw[c]  = pd.to_numeric(X_test_raw[c], errors="coerce")

# 3) Guardrail: nếu vẫn còn object trong numeric -> fail sớm để biết cột nào
still_bad = [c for c in NUMERIC_FEATURES if c in X_train_raw.columns and X_train_raw[c].dtype == "object"]
assert len(still_bad) == 0, f"Still non-numeric in NUMERIC_FEATURES: {still_bad}"
print("✅ Numeric columns are clean now.")


After drop: (2345597, 34) (502628, 34) (502628, 34)
Numeric cols with object dtype: ['src_bytes']
✅ Numeric columns are clean now.


In [16]:
preprocess.fit(X_train_raw)

X_train = preprocess.transform(X_train_raw)
X_val   = preprocess.transform(X_val_raw)
X_test  = preprocess.transform(X_test_raw)

final_feature_names = preprocess.get_feature_names_out().tolist()

print("Transformed shapes:", X_train.shape, X_val.shape, X_test.shape)
print("Total final features:", len(final_feature_names))
print("First 30 features:", final_feature_names[:30])



Transformed shapes: (2345597, 104) (502628, 104) (502628, 104)
Total final features: 104
First 30 features: ['duration', 'src_bytes', 'dst_bytes', 'src_pkts', 'dst_pkts', 'src_ip_bytes', 'dst_ip_bytes', 'missed_bytes', 'src_port', 'dst_port', 'http_request_body_len', 'http_response_body_len', 'proto_icmp', 'proto_tcp', 'proto_udp', 'service_NONE', 'service_dce_rpc', 'service_dce_rpc;ntlm', 'service_dhcp', 'service_dns', 'service_ftp', 'service_ftp-data', 'service_gssapi', 'service_gssapi;ntlm', 'service_gssapi;ntlm;smb', 'service_gssapi;smb', 'service_gssapi;smb;ntlm', 'service_http', 'service_imap', 'service_imap;ssl']


In [19]:
OUT_DIR = "../outputs/preprocess"
os.makedirs(OUT_DIR, exist_ok=True)

# Save preprocess pipeline
joblib.dump(preprocess, os.path.join(OUT_DIR, "preprocess.pkl"))

# Save feature names in correct order
with open(os.path.join(OUT_DIR, "feature_names.json"), "w", encoding="utf-8") as f:
    json.dump(final_feature_names, f, indent=2)

# Save schema/spec (helps thesis & stage 3)
schema = {
    "drop_columns": DROP_COLUMNS,
    "numeric_features": NUMERIC_FEATURES,
    "categorical_features": CATEGORICAL_FEATURES,
    "boolean_features": BOOLEAN_FEATURES,
    "final_feature_count": len(final_feature_names),
}
with open(os.path.join(OUT_DIR, "feature_schema.json"), "w", encoding="utf-8") as f:
    json.dump(schema, f, indent=2)

print("Saved artifacts to:", OUT_DIR)
print("- preprocess.pkl")
print("- feature_names.json")
print("- feature_schema.json")

Saved artifacts to: ../outputs/preprocess
- preprocess.pkl
- feature_names.json
- feature_schema.json


In [20]:
# Re-transform train and confirm feature names length matches
assert X_train.shape[1] == len(final_feature_names), "Mismatch feature dimension vs names"

# Quick check that transformed arrays are finite
assert np.isfinite(X_train).all(), "Non-finite values in X_train"
assert np.isfinite(X_val).all(), "Non-finite values in X_val"
assert np.isfinite(X_test).all(), "Non-finite values in X_test"

print("Sanity checks passed: feature order & numeric stability OK.")


Sanity checks passed: feature order & numeric stability OK.


In [21]:
OUT_DIR = "../outputs/processed"
os.makedirs(OUT_DIR, exist_ok=True)

# đảm bảo X_* là numpy array float32 cho nhẹ + ổn định
X_train_np = np.asarray(X_train, dtype=np.float32)
X_val_np   = np.asarray(X_val,   dtype=np.float32)
X_test_np  = np.asarray(X_test,  dtype=np.float32)

y_train_np = np.asarray(y_train, dtype=np.int64)
y_val_np   = np.asarray(y_val,   dtype=np.int64)
y_test_np  = np.asarray(y_test,  dtype=np.int64)

np.save(os.path.join(OUT_DIR, "X_train.npy"), X_train_np)
np.save(os.path.join(OUT_DIR, "X_val.npy"),   X_val_np)
np.save(os.path.join(OUT_DIR, "X_test.npy"),  X_test_np)

np.save(os.path.join(OUT_DIR, "y_train.npy"), y_train_np)
np.save(os.path.join(OUT_DIR, "y_val.npy"),   y_val_np)
np.save(os.path.join(OUT_DIR, "y_test.npy"),  y_test_np)

# lưu feature order đúng thứ tự (rất quan trọng cho Stage 3)
with open(os.path.join(OUT_DIR, "feature_order.json"), "w", encoding="utf-8") as f:
    json.dump(final_feature_names, f, indent=2)

print("✅ Saved arrays to", OUT_DIR)
print("X_train:", X_train_np.shape, "X_val:", X_val_np.shape, "X_test:", X_test_np.shape)


✅ Saved arrays to ../outputs/processed
X_train: (2345597, 104) X_val: (502628, 104) X_test: (502628, 104)
